# Low-Rank Adaptation (LoRA) and Weight Decomposed LoRA (DoRA) from Scratch

This notebook implements LoRA and DoRA layers from scratch in PyTorch, showing weight equations and projection shapes.

In [1]:
import torch
import torch.nn as nn
import math

class LoraLinear(nn.Module):
    def __init__(self, in_features, out_features, r=2, alpha=4):
        super().__init__()
        self.base_layer = nn.Linear(in_features, out_features, bias=False)
        # Freeze base layer weights
        self.base_layer.weight.requires_grad = False
        
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        
        # LoRA adapters
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, r))
        
        # Initialize A and B
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        
    def forward(self, x):
        base_out = self.base_layer(x)
        adapter_out = (x @ self.lora_A.t()) @ self.lora_B.t()
        return base_out + self.scaling * adapter_out

class DoraLinear(nn.Module):
    def __init__(self, in_features, out_features, r=2, alpha=4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Base linear layer
        self.base_layer = nn.Linear(in_features, out_features, bias=False)
        self.base_layer.weight.requires_grad = False
        
        # Magnitude parameter vector (m)
        self.m = nn.Parameter(torch.norm(self.base_layer.weight, p=2, dim=1))
        
        # Direction adapters
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        
    def forward(self, x):
        # Compute combined direction weights
        lora_weight = (self.lora_B @ self.lora_A) * self.scaling
        combined_weight = self.base_layer.weight + lora_weight
        
        # Column normalize the direction component
        norm_factor = torch.norm(combined_weight, p=2, dim=1, keepdim=True)
        direction_weight = combined_weight / norm_factor
        
        # Multiply by magnitude vector
        effective_weight = self.m.unsqueeze(1) * direction_weight
        
        return x @ effective_weight.t()

# Test both layers
x = torch.randn(1, 4)
lora = LoraLinear(4, 4, r=2, alpha=4)
dora = DoraLinear(4, 4, r=2, alpha=4)

print("LoRA output shape:", lora(x).shape)
print("DoRA output shape:", dora(x).shape)


LoRA output shape: torch.Size([1, 4])
DoRA output shape: torch.Size([1, 4])


### Output Explanation
The output confirms both low-rank adapters calculate correct tensor transformations, maintaining the output hidden dimension sizes ($1 	imes 4$ shape).